In [15]:
import os
import duckdb
import pandas as pd

# Hardcode the project root — no ambiguity, always works
PROJECT_ROOT = r"D:\olist-analytics"
DATA_PATH = r"D:\olist-analytics\data\raw"

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

con = duckdb.connect(":memory:")
print("DuckDB connected.")

Working directory: D:\olist-analytics
DuckDB connected.


In [16]:
con.execute(f"""
    CREATE TABLE orders AS 
    SELECT * FROM read_csv_auto('{DATA_PATH}/olist_orders_dataset.csv')
""")

con.execute(f"""
    CREATE TABLE customers AS 
    SELECT * FROM read_csv_auto('{DATA_PATH}/olist_customers_dataset.csv')
""")

con.execute(f"""
    CREATE TABLE order_items AS 
    SELECT * FROM read_csv_auto('{DATA_PATH}/olist_order_items_dataset.csv')
""")

con.execute(f"""
    CREATE TABLE order_payments AS 
    SELECT * FROM read_csv_auto('{DATA_PATH}/olist_order_payments_dataset.csv')
""")

con.execute(f"""
    CREATE TABLE products AS 
    SELECT * FROM read_csv_auto('{DATA_PATH}/olist_products_dataset.csv')
""")

con.execute(f"""
    CREATE TABLE sellers AS 
    SELECT * FROM read_csv_auto('{DATA_PATH}/olist_sellers_dataset.csv')
""")

print("All tables loaded.")
con.execute("SELECT COUNT(*) as total_orders FROM orders").fetchdf()

All tables loaded.


,total_orders
0,99441


In [17]:
con.execute("""
    CREATE TABLE fact_orders AS
    SELECT
        o.order_id,
        c.customer_unique_id,
        o.order_status,
        CAST(o.order_purchase_timestamp AS TIMESTAMP)   AS order_date,
        CAST(o.order_delivered_customer_date AS TIMESTAMP) AS delivery_date,
        DATE_DIFF('day',
            CAST(o.order_purchase_timestamp AS TIMESTAMP),
            CAST(o.order_delivered_customer_date AS TIMESTAMP)
        ) AS delivery_days,
        COALESCE(p.total_payment, 0)        AS total_payment,
        COALESCE(p.payment_installments, 1) AS installments,
        COALESCE(i.item_count, 0)           AS item_count
    FROM orders o
    LEFT JOIN (
        SELECT order_id,
               SUM(payment_value)          AS total_payment,
               MAX(payment_installments)   AS payment_installments
        FROM order_payments
        GROUP BY order_id
    ) p ON o.order_id = p.order_id
    LEFT JOIN (
        SELECT order_id, COUNT(*) AS item_count
        FROM order_items
        GROUP BY order_id
    ) i ON o.order_id = i.order_id
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
""")

print("fact_orders created.")

con.execute("""
    SELECT COUNT(*) as rows,
           ROUND(AVG(total_payment), 2)  as avg_order_value,
           ROUND(AVG(delivery_days), 1)  as avg_delivery_days
    FROM fact_orders
""").fetchdf()

fact_orders created.


,rows,avg_order_value,avg_delivery_days
0,96478,159.85,12.5


In [18]:
monthly_revenue = con.execute("""
    SELECT
        DATE_TRUNC('month', order_date)    AS month,
        COUNT(DISTINCT order_id)           AS orders,
        COUNT(DISTINCT customer_unique_id) AS unique_customers,
        ROUND(SUM(total_payment), 2)       AS revenue,
        ROUND(AVG(total_payment), 2)       AS avg_order_value,
        ROUND(
            100.0 * (SUM(total_payment) - LAG(SUM(total_payment))
                     OVER (ORDER BY DATE_TRUNC('month', order_date)))
            / NULLIF(LAG(SUM(total_payment))
                     OVER (ORDER BY DATE_TRUNC('month', order_date)), 0)
        , 1) AS mom_growth_pct
    FROM fact_orders
    GROUP BY DATE_TRUNC('month', order_date)
    ORDER BY month
""").fetchdf()

monthly_revenue

,month,orders,unique_customers,revenue,avg_order_value,mom_growth_pct
0,2016-09-01,1,1,0.00,0.00,NaN
1,2016-10-01,265,262,46566.71,175.72,NaN
2,2016-12-01,1,1,19.62,19.62,-100.0
3,2017-01-01,750,718,127545.67,170.06,649979.9
4,2017-02-01,1653,1630,271298.65,164.13,112.7
5,2017-03-01,2546,2508,414369.39,162.75,52.7
6,2017-04-01,2303,2274,390952.18,169.76,-5.7
7,2017-05-01,3546,3479,567066.73,159.92,45.0
8,2017-06-01,3135,3076,490225.60,156.37,-13.6
9,2017-07-01,3872,3802,566403.93,146.28,15.5


In [19]:
top_categories = con.execute("""
    SELECT
        COALESCE(p.product_category_name, 'unknown') AS category,
        COUNT(DISTINCT oi.order_id)   AS orders,
        ROUND(SUM(oi.price), 2)       AS revenue,
        ROUND(AVG(oi.price), 2)       AS avg_price
    FROM order_items oi
    LEFT JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN orders o   ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY p.product_category_name
    ORDER BY revenue DESC
    LIMIT 20
""").fetchdf()

top_categories

,category,orders,revenue,avg_price
0,beleza_saude,8647,1233131.72,130.28
1,relogios_presentes,5495,1166176.98,199.04
2,cama_mesa_banho,9272,1023434.76,93.44
3,esporte_lazer,7530,954852.55,113.25
4,informatica_acessorios,6530,888724.61,116.26
5,moveis_decoracao,6307,711927.69,87.25
6,utilidades_domesticas,5743,615628.69,90.60
7,cool_stuff,3559,610204.10,164.12
8,automotivo,3810,578966.65,139.85
9,brinquedos,3804,471286.48,116.94


In [20]:
repeat_customers = con.execute("""
    WITH customer_orders AS (
        SELECT
            customer_unique_id,
            COUNT(DISTINCT order_id) AS total_orders,
            MIN(order_date)          AS first_order,
            MAX(order_date)          AS last_order,
            SUM(total_payment)       AS lifetime_value
        FROM fact_orders
        GROUP BY customer_unique_id
    )
    SELECT
        CASE
            WHEN total_orders = 1  THEN '1 - one-time buyer'
            WHEN total_orders = 2  THEN '2 - bought twice'
            WHEN total_orders >= 3 THEN '3+ - loyal customer'
        END AS customer_segment,
        COUNT(*)                      AS customer_count,
        ROUND(AVG(lifetime_value), 2) AS avg_ltv,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_customers
    FROM customer_orders
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

repeat_customers

,customer_segment,customer_count,avg_ltv,pct_of_customers
0,1 - one-time buyer,90557,160.76,97.0
1,2 - bought twice,2573,291.03,2.8
2,3+ - loyal customer,228,506.79,0.2


In [ ]:
# KEY FINDINGS

# 1. SCALE: 96,478 delivered orders | Avg order R$159.85 | Avg delivery 12.5 days
#    Note: ~3,000 orders excluded (not delivered) — worth investigating status breakdown

# 2. SEASONALITY: Clear Black Friday spike Nov 2017 (+53.6% MoM)
#    Recommendation: pre-build inventory in Oct, run early-access campaigns

# 3. TOP CATEGORY: beleza_saude (#1 revenue) vs pcs (#1 avg price at R$1,099)
#    High-ticket categories need different strategy — fewer orders, more value per sale

# 4. RETENTION CRISIS: 97% of customers never return
#    BUT: repeat buyers are worth 2-3x more (R$160 vs R$291 vs R$506 LTV)
#    Recommendation: focus on month-1 to month-2 conversion — highest ROI opportunity